# Multilingual Health Q&A — Starter Notebook

**Challenge:** Multilingual Health Question Answering in Low-Resource African Languages

This notebook provides two baselines for the challenge:

| Baseline | Approach | Speed | Quality |
|---|---|---|---|
| **Baseline 1** | TF-IDF retrieval | Very fast | Low |
| **Baseline 2** | Multilingual LLM (mT5 / NLLB) | Moderate | Higher |

**Task:** Given a health question in one of five African languages (Amharic, Luganda, Akan, Swahili, or English), generate a fluent and accurate answer in the **same language**.

**Evaluation metrics:**
- `TargetRLF1` — ROUGE-L F1 score
- `TargetR1F1` — ROUGE-1 F1 score
- `TargetLLM`  — LLM-as-a-Judge score

> **Note:** All three submission columns should contain the same generated answer for each row. The platform computes all three metrics from that single answer.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
# Install required packages
!pip install -q scikit-learn pandas numpy rouge-score
!pip install -q transformers sentencepiece accelerate torch

print('✅ Packages installed')

  Preparing metadata (setup.py) ... done
✅ Packages installed


In [3]:
import re
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', None)

print('✅ Imports complete')

✅ Imports complete


## 2 — Set File Paths

In [4]:
DATA_DIR = Path('/content')

TRAIN_PATH      = DATA_DIR / 'Train.csv'
TEST_PATH       = DATA_DIR / 'Test.csv'
VAL_PATH        = DATA_DIR / 'Val.csv'
SAMPLE_SUB_PATH = DATA_DIR / 'SampleSubmission.csv'

# Output paths for each baseline
OUTPUT_TFIDF = Path('/content/submission_tfidf_baseline.csv')
OUTPUT_LLM   = Path('/content/submission_llm_baseline.csv')

for path in [TRAIN_PATH, TEST_PATH, VAL_PATH, SAMPLE_SUB_PATH]:
    status = '✅' if path.exists() else '❌'
    print(f'{status} {path}')

✅ /content/Train.csv
✅ /content/Test.csv
✅ /content/Val.csv
✅ /content/SampleSubmission.csv


## 3 — Load and Preview the Data

In [5]:
train             = pd.read_csv(TRAIN_PATH)
test              = pd.read_csv(TEST_PATH)
val               = pd.read_csv(VAL_PATH)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

print(f'Train shape             : {train.shape}')
print(f'Test shape              : {test.shape}')
print(f'Val shape               : {val.shape}')
print(f'Sample submission shape : {sample_submission.shape}')
print()
print('Train columns:', train.columns.tolist())
print('Test columns :', test.columns.tolist())
print('Val columns  :', val.columns.tolist())

display(train.head(3))
display(test.head(3))
display(sample_submission.head(3))

Train shape             : (29815, 4)
Test shape              : (2618, 3)
Val shape               : (6686, 4)
Sample submission shape : (2618, 4)

Train columns: ['ID', 'input', 'output', 'subset']
Test columns : ['ID', 'input', 'subset']
Val columns  : ['ID', 'input', 'output', 'subset']


,ID,input,output,subset
0,ID_TR_Aka_Gha_A3B1799D,Ɔkwan bɛn so na mmabunbɛtumi aboa wɔn mfɛfoɔ a nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ? Yei bi ne sɛnea wɔbɛkyekye w...,Mmabun betumi aboa atipɛnfo a ebia nsa anaa nnubɔne ama wɔayɛ wɔn ayayadeɛ so denam: Nkate fam mmoa a wɔde bɛma na w...,Aka_Gha
1,ID_TR_Aka_Gha_1C80317F,"Edinnsiananmu bɛn na nnipa a ɛsono wɔn bɔbeasu taa de di dwuma, na yɛbɛyɛ dɛn ahwɛ ahu sɛ yɛde redi dwuma yiye?","Wɔ Ghana mu no, amanmmra no gye binary gender nkutoo tom a she/he edinnsiananmu nkutoo na ɛka ho",Aka_Gha
2,ID_TR_Aka_Gha_06671AD1,Ɔkwan bɛn so na ɔbarima ne ɔbea nna a wɔtwe wɔn ho fi ho anaa nna mu adwumadi a wɔtwentwɛn so no boa ma asiane so tew?,"Sɛ wɔtwe wɔn ho fi nna mu anaasɛ wɔtwentwɛn wɔn nan ase a, ɛboa ma asiane nso tew denam asiane a ɛwɔ STI ne HIV a ɛb...",Aka_Gha


,ID,input,subset
0,ID_TS_Aka_Gha_A3B1799D,"Fa nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow a wɔreyɛ adwuma de asiw GBV ano ma.",Aka_Gha
1,ID_TS_Aka_Gha_1C80317F,Dɛn ne nea ebetumi afi hokwan a mmabun wɔ sɛ wonya nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu a wobu...,Aka_Gha
2,ID_TS_Aka_Gha_06671AD1,Akwan bɛn na mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi aba ɛfa nnipa a wɔbɛgyina ho kɛkɛ 'bystander' wɔ bere a as...,Aka_Gha


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
1,ID_TS_Aka_Gha_1C80317F,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
2,ID_TS_Aka_Gha_06671AD1,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"


In [6]:
# Explore language distribution in training data
print('Language distribution in training set:')
display(train['subset'].value_counts())

Language distribution in training set:


,count
subset,
Eng_Uga,7624
Aka_Gha,4455
Eng_Gha,4443
Eng_Eth,3915
Lug_Uga,3383
Eng_Ken,2080
Swa_Ken,2070
Amh_Eth,1845


In [7]:
ID_COL           = 'ID'
TEST_ID_COL      = 'ID'
QUESTION_COL     = 'input'
TEST_QUESTION_COL= 'input'
ANSWER_COL       = 'output'
LANG_COL         = 'subset'
TEST_LANG_COL    = 'subset'

print(f'  Train ID        : {ID_COL}')
print(f'  Test ID         : {TEST_ID_COL}')
print(f'  Train question  : {QUESTION_COL}')
print(f'  Test question   : {TEST_QUESTION_COL}')
print(f'  Train answer    : {ANSWER_COL}')
# ── Language code mapping ──────────────────────────────────────────────────────
# The `subset` column encodes language and country as "<LangCode>_<CountryCode>".
# Only the first part identifies the language; the second is the country.
# This dictionary maps the language prefix to a full language name used in prompts.
SUBSET_TO_LANGUAGE = {
    'Eng': 'English',
    'Aka': 'Akan',
    'Lug': 'Luganda',
    'Swa': 'Swahili',
    'Amh': 'Amharic',
}

def subset_to_language_name(subset_code: str) -> str:
    """
    Extract the full language name from a subset code such as 'Amh_Eth' or 'Aka_Gha'.
    Falls back to the raw code if the prefix is not recognised.
    """
    if not subset_code or not isinstance(subset_code, str):
        return 'English'
    lang_prefix = subset_code.split('_')[0]
    return SUBSET_TO_LANGUAGE.get(lang_prefix, subset_code)

print('Language mapping:')
for code, name in SUBSET_TO_LANGUAGE.items():
    print(f'  {code}_* → {name}')


  Train ID        : ID
  Test ID         : ID
  Train question  : input
  Test question   : input
  Train answer    : output
Language mapping:
  Eng_* → English
  Aka_* → Akan
  Lug_* → Luganda
  Swa_* → Swahili
  Amh_* → Amharic


## 5 — Text Cleaning

In [8]:
def clean_text(x):
    """Strip whitespace and handle null values."""
    if pd.isna(x):
        return ''
    return str(x).strip()

train[QUESTION_COL]      = train[QUESTION_COL].map(clean_text)
train[ANSWER_COL]        = train[ANSWER_COL].map(clean_text)
test[TEST_QUESTION_COL]  = test[TEST_QUESTION_COL].map(clean_text)
val[QUESTION_COL]        = val[QUESTION_COL].map(clean_text)
val[ANSWER_COL]          = val[ANSWER_COL].map(clean_text)

# Remove rows with empty questions or answers
train = train[(train[QUESTION_COL] != '') & (train[ANSWER_COL] != '')].reset_index(drop=True)
test  = test[test[TEST_QUESTION_COL]  != ''].reset_index(drop=True)
val   = val[(val[QUESTION_COL] != '') & (val[ANSWER_COL] != '')].reset_index(drop=True)

print(f'Cleaned train shape : {train.shape}')
print(f'Cleaned test shape  : {test.shape}')
print(f'Cleaned val shape   : {val.shape}')

Cleaned train shape : (29814, 4)
Cleaned test shape  : (2618, 3)
Cleaned val shape   : (6686, 4)


## 7 — Evaluation Utilities

ROUGE-1 and ROUGE-L scoring using whitespace tokenisation — safe for non-English scripts.

In [9]:
try:
    from rouge_score import rouge_scorer

    class WhitespaceTokenizer:
        """Whitespace tokeniser — language-agnostic and safe for African scripts."""
        def tokenize(self, text):
            if text is None:
                return []
            return str(text).strip().split()

    def compute_rouge(predictions, references):
        """
        Compute mean ROUGE-1 and ROUGE-L F1 scores.

        Parameters
        ----------
        predictions : list[str]
        references  : list[str]

        Returns
        -------
        dict with rouge1_f1 and rougeL_f1
        """
        scorer = rouge_scorer.RougeScorer(
            ['rouge1', 'rougeL'],
            tokenizer    = WhitespaceTokenizer(),
            use_stemmer  = False,
        )
        r1_scores, rl_scores = [], []

        for pred, ref in zip(predictions, references):
            score = scorer.score(str(ref), str(pred))
            r1_scores.append(score['rouge1'].fmeasure)
            rl_scores.append(score['rougeL'].fmeasure)

        return {
            'rouge1_f1': float(np.mean(r1_scores)) if r1_scores else 0.0,
            'rougeL_f1': float(np.mean(rl_scores)) if rl_scores else 0.0,
        }

    def compute_rouge_by_language(predictions, references, languages):
        """Compute ROUGE scores broken down by language."""
        results = {}
        lang_arr = np.array(languages)

        for lang in np.unique(lang_arr):
            mask    = lang_arr == lang
            preds_l = [p for p, m in zip(predictions, mask) if m]
            refs_l  = [r for r, m in zip(references,  mask) if m]
            results[lang] = compute_rouge(preds_l, refs_l)

        return pd.DataFrame(results).T

    print('✅ ROUGE scorer loaded')

except ImportError:
    print('⚠️  rouge-score not installed. Run: pip install rouge-score')
    compute_rouge = None

✅ ROUGE scorer loaded


## 8 — Baseline 1: TF-IDF Retrieval

For each test question, find the most similar training question using TF-IDF character n-grams and return its answer.

**Why character n-grams?** Character-level features work across scripts (Latin, Amharic Ge'ez, etc.) without requiring language-specific tokenisation.

In [10]:
class TfidfRetrievalAnswerer:
    """
    TF-IDF nearest-neighbour retrieval baseline.

    Builds a per-language model if a group column is available,
    falling back to a global model for unseen groups.
    """

    def __init__(self, question_col, answer_col, group_col=None,
                 ngram_range=(3, 5), max_features=200_000):
        self.question_col = question_col
        self.answer_col   = answer_col
        self.group_col    = group_col
        self.ngram_range  = ngram_range
        self.max_features = max_features
        self.models       = {}
        self.global_model = None

    def _fit_single(self, df):
        """Fit a vectoriser and nearest-neighbour index on a subset."""
        vectorizer = TfidfVectorizer(
            analyzer     = 'char_wb',
            ngram_range  = self.ngram_range,
            min_df       = 1,
            max_features = self.max_features,
            lowercase    = False,   # preserve case for non-Latin scripts
        )
        questions = df[self.question_col].fillna('').astype(str).tolist()
        answers   = df[self.answer_col].fillna('').astype(str).tolist()

        X  = vectorizer.fit_transform(questions)
        nn = NearestNeighbors(n_neighbors=1, metric='cosine')
        nn.fit(X)

        return {
            'vectorizer': vectorizer,
            'nn'        : nn,
            'answers'   : np.array(answers,   dtype=object),
            'questions' : np.array(questions, dtype=object),
        }

    def fit(self, df):
        """Fit the global model and per-group models."""
        self.global_model = self._fit_single(df)
        if self.group_col and self.group_col in df.columns:
            for group, sub in df.groupby(self.group_col):
                if len(sub) >= 2:
                    self.models[group] = self._fit_single(sub)
        print(f'  Fitted global model + {len(self.models)} group model(s)')
        return self

    def _predict_one_from_model(self, question, model):
        Xq        = model['vectorizer'].transform([question])
        dist, idx = model['nn'].kneighbors(Xq, n_neighbors=1)
        i         = idx[0][0]
        sim       = 1 - float(dist[0][0])
        return model['answers'][i], sim, model['questions'][i]

    def predict_one(self, question, group=None):
        model = self.models.get(group, self.global_model) if group is not None else self.global_model
        return self._predict_one_from_model(question, model)

    def predict(self, df, question_col, group_col=None):
        outputs, similarities, matched = [], [], []
        for _, row in df.iterrows():
            question = clean_text(row[question_col])
            group    = row[group_col] if group_col and group_col in df.columns else None
            answer, sim, matched_q = self.predict_one(question, group)
            outputs.append(answer)
            similarities.append(sim)
            matched.append(matched_q)
        return outputs, similarities, matched

print('✅ TfidfRetrievalAnswerer defined')

✅ TfidfRetrievalAnswerer defined


In [11]:
# Choose grouping strategy — prefer config (language+country), else language
GROUP_COL      =  LANG_COL
TEST_GROUP_COL =  TEST_LANG_COL

print(f'Group column      : {GROUP_COL}')
print(f'Test group column : {TEST_GROUP_COL}')

Group column      : subset
Test group column : subset


In [12]:
# ── Validate TF-IDF baseline on the local validation set ──────────────────────
print('Training TF-IDF retrieval baseline on training partition...')

answerer_valid = TfidfRetrievalAnswerer(
    question_col = QUESTION_COL,
    answer_col   = ANSWER_COL,
    group_col    = GROUP_COL,
).fit(train)

valid_pred, valid_sim, valid_match = answerer_valid.predict(
    val,
    question_col = QUESTION_COL,
    group_col    = GROUP_COL,
)

if compute_rouge:
    metrics = compute_rouge(valid_pred, val[ANSWER_COL].tolist())
    print(f'\n📊 TF-IDF Baseline — Validation ROUGE Scores')
    print(f'   ROUGE-1 F1 : {metrics["rouge1_f1"]:.4f}')
    print(f'   ROUGE-L F1 : {metrics["rougeL_f1"]:.4f}')

    # Break down by language
    if LANG_COL and LANG_COL in val.columns:
        print('\n📊 ROUGE scores by language:')
        lang_metrics = compute_rouge_by_language(
            valid_pred,
            val[ANSWER_COL].tolist(),
            val[LANG_COL].tolist()
        )
        display(lang_metrics.round(4))

# Preview
preview = val[[ID_COL, QUESTION_COL, ANSWER_COL]].copy()
preview['baseline_answer']        = valid_pred
preview['retrieval_similarity']   = [f'{s:.3f}' for s in valid_sim]
preview['matched_train_question'] = valid_match
display(preview.head(5))

Training TF-IDF retrieval baseline on training partition...
  Fitted global model + 8 group model(s)

📊 TF-IDF Baseline — Validation ROUGE Scores
   ROUGE-1 F1 : 0.4207
   ROUGE-L F1 : 0.3655

📊 ROUGE scores by language:


,rouge1_f1,rougeL_f1
Aka_Gha,0.2832,0.1674
Amh_Eth,0.1455,0.1353
Eng_Eth,0.5170,0.4994
Eng_Gha,0.2582,0.1707
Eng_Ken,0.5989,0.5606
Eng_Uga,0.5163,0.4709
Lug_Uga,0.5155,0.4935
Swa_Ken,0.6031,0.5672


,ID,input,output,baseline_answer,retrieval_similarity,matched_train_question
0,ID_VL_Aka_Gha_A3B1799D,"Sɛn na nwomasua ne adwuma nteteeɛ boa akuo a eye mmabun a wɔ hia neaɛma sokoronko ne ohaw ahorow, atubrafo, anaa wɔn...",Nhyehyɛeɛ aa ama ne mu so te sɛ senea aborɔfo ka no 'STEM' ne 'vocational training' se ɛbɛ adrɛse mmabun kuokuo ahoh...,"Nkɔmmɔdi: Aban, NGO, ne amanaman ntam nnwumakuo ntam nkitahodiɛ bɛtumi ama nkitahodiɛ, nneɛma a wɔboaboa ano, ne dwu...",0.378,"Ɔkwan bɛn so na aban ahorow, ahyehyɛde ahorow a ɛnyɛ aban de (NGOs), ne amanaman ntam nnwumakuw ne fekubɔ betumi ahy..."
1,ID_VL_Aka_Gha_1C80317F,Dɛn nti na ɛho hia sɛ mmabun te wɔn nna ne awo hokwan ahorow ase?,"Nna ne awo hokwan ahorow a wɔte ase no ma mmabun tumi: Si gyinae a ɛfata wɔ wɔn nipadua, nna, ne abusuabɔ ho. Kamfo ...",Mmara kwan so hokwan ahorow kyerɛ hokwan ahorow ne ahobammɔ a mmara de ma ankorankoro. Wɔ nna ne awo akwahosan ho no...,0.545,"Dɛn ne mmara kwan so ahokwan ahorow, na dɛn nti na ɛho hia sɛ mmabun hu ho wɔ nna ne awoɔ akwahosan ho?"
2,ID_VL_Aka_Gha_06671AD1,"Mɛyɛ dɛn atumi abɔ asisifo ho amanneɛ wɔ ɔkwan a etu mpɔn na ahobammɔ wom so, na anammɔn bɛn na metumi atu de ahwɛ a...",Ayayade ho nsɛm a wɔbɛbɔ ho amanneɛ yiye na ahobammɔ wom no hwehwɛ sɛ wɔyɛ nneɛma a edidi so yi: Wɔkyerɛw ayayade no...,"Akwan ahorow: Siesie Wo Ho Di Kan: Kyerɛw wo nsɛmmisa ne wo dadwen to hɔ ansa na woakɔ. Di Nokware: Kasa fa wo nna, ...",0.292,Akwan bɛn so na metumi ne dɔkotafo anaa nɛɛse akasa afa me awoɔ mu apɔmuden ho na mahwɛ sɛ wɔate m’ahiadeɛ ne me dad...
3,ID_VL_Aka_Gha_BDD640FB,"Ɔkwan bɛn so na mmabun betumi de akwan ahorow a egyina nnipa hokwan ahorow so adi dwuma, de akamfo nna ho nkyerɛkyer...","Mmabun betumi de akwan a, egyina nnipa hokwan ahorow so adi dwuma de akamfo nna ho nkyerɛkyerɛ a edi mũ wɔ sukuu ne ...",Mmabun betumi akasa ama nna ho nkyerɛkyerɛ a edi mu wɔ wɔn sukuu ne mpɔtam hɔ denam: Akuo a sukuufo di anim a wɔde w...,0.642,Ɔkwan bɛn so na mmabun betumi akasa afa nna ho nkyerɛkyerɛ a edi mu wɔ wɔn sukuu ne mpɔtam hɔ?
4,ID_VL_Aka_Gha_46685257,"Dwuma bɛn na abɛɛfo mfidie 'Technology' di de boa mmabun ma wonya ɔhwɛ a ɛfata, a telehealth ne online resources ka ho?",Abɛɛfo mfidie te sɛ telehealth ma mmabun kwan ma wonya ɔhwɛ fi wɔn fie. Wei yi akwansidie te sɛ lore a wobɛfo ɛnam b...,Websites (nimdeɛ pa). Apps (de hwehwɛ nneɛma). SMS (nkɔmmɔbɔ mfidie). Telemedicine (intanɛt so ayaresa).,0.412,Sɛn na abɛɛfo mfidie 'technology' boa SRH nimdeɛ ho nkyerɛkyerɛ a ɛkɔ so wɔ wiase nyinaa?


In [13]:
# ── Train on all data and predict test answers ─────────────────────────────────
print('Training TF-IDF retrieval on full training set...')

answerer = TfidfRetrievalAnswerer(
    question_col = QUESTION_COL,
    answer_col   = ANSWER_COL,
    group_col    = GROUP_COL,
).fit(train)

test_pred_tfidf, test_sim, test_match = answerer.predict(
    test,
    question_col = TEST_QUESTION_COL,
    group_col    = TEST_GROUP_COL,
)

print(f'Generated {len(test_pred_tfidf)} predictions')

preview = test[[TEST_ID_COL, TEST_QUESTION_COL]].copy()
preview['baseline_answer']      = test_pred_tfidf
preview['retrieval_similarity'] = [f'{s:.3f}' for s in test_sim]
display(preview.head(5))

Training TF-IDF retrieval on full training set...
  Fitted global model + 8 group model(s)
Generated 2618 predictions


,ID,input,baseline_answer,retrieval_similarity
0,ID_TS_Aka_Gha_A3B1799D,"Fa nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow a wɔreyɛ adwuma de asiw GBV ano ma.","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa intanɛt so nneɛma a efi nnwumakuo ahorow a wogye wɔn di mu a ɛfa GBV n...",0.464
1,ID_TS_Aka_Gha_1C80317F,Dɛn ne nea ebetumi afi hokwan a mmabun wɔ sɛ wonya nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu a wobu...,"Yiw, mmabun betumi ahwehwɛ mmara kwan so mmoa sɛ wogye di sɛ wɔabu hokwan a wɔwɔ sɛ wodi wɔn ho so wɔ nipadua mu no ...",0.451
2,ID_TS_Aka_Gha_06671AD1,Akwan bɛn na mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi aba ɛfa nnipa a wɔbɛgyina ho kɛkɛ 'bystander' wɔ bere a as...,Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basabasayɛ kwan denam mmoa a wɔde ma (bystanders) ma wohu/yɛ biribi no so. ...,0.250
3,ID_TS_Aka_Gha_BDD640FB,"Sɛnea amammerɛ mu mmra, asetena mu suban, ne tumi mu nsakraeɛ a ɛwɔ nkuro ne atuhoamafoɔ mu ka mmaabun nkitahodi ne ...","Hokwan a mmabun akannifoɔ de wɔn ho bɛhyɛ dwumadiɛ a ɛma wɔn tumi yɛ adwuma, afotuo nhyehyɛeɛ, ne atipɛnfoɔ adesua n...",0.357
4,ID_TS_Aka_Gha_46685257,Adɛn nti na mmara nsesaeɛ 'policy advocacy' ho hia ma mmabun nyin wɔ biribiara mu ?,Kae sɛ obiara suahunu yɛ soronko. Dwen sɛ obiara wɔ n'adwene kyerɛ. Gye tom sɛ amammerɛ ne nyamesom sesa nkurɔfoɔ su...,0.361


## 9 — Baseline 2: Multilingual LLM (mT5 / AfroLM)

This baseline uses a pre-trained multilingual sequence-to-sequence model to generate answers directly, without retrieval.

**Model options (choose one):**

| Model | Languages | Size | Notes |
|---|---|---|---|
| `google/mt5-small` | 101 languages incl. Swahili | 300M | Fast, good multilingual coverage |
| `google/mt5-base` | 101 languages | 580M | Better quality, slower |
| `facebook/nllb-200-distilled-600M` | 200 languages incl. Luganda, Akan | 600M | Best African language coverage |
| `Helsinki-NLP/opus-mt-mul-en` | Many → English | 300M | English output only |

> **Recommendation:** Start with `google/mt5-small` for speed, then try `facebook/nllb-200-distilled-600M` for better coverage of low-resource languages.

**For a strong fine-tuned solution:**
Fine-tune `google/mt5-base` or `facebook/nllb-200-distilled-600M` on the provided training data using the Hugging Face `Seq2SeqTrainer`. See the fine-tuning section at the end of this notebook.

In [14]:
import torch
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── Configuration ──────────────────────────────────────────────────────────────
MODEL_NAME = 'google/mt5-small'
# MODEL_NAME = 'google/mt5-base'
# MODEL_NAME = 'facebook/nllb-200-distilled-600M'

MAX_INPUT_LENGTH  = 128
MAX_OUTPUT_LENGTH = 256
BATCH_SIZE_LLM    = 2
NUM_BEAMS         = 4

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')
print(f'Model   : {MODEL_NAME}')

if DEVICE == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device  : cuda
Model   : google/mt5-small
GPU     : Tesla T4
VRAM    : 15.6 GB


In [15]:
# ── Load the model and tokeniser ───────────────────────────────────────────────
print(f'Loading {MODEL_NAME}...')
print('This may take a few minutes on first run (downloading model weights).')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_llm = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    # Always load in float32 so gradient computation stays in float32.
    # fp16/bf16 mixed precision is handled by the Trainer via grad scaler,
    # not by storing the model weights in float16 directly.
    torch_dtype = torch.float32,
)
model_llm = model_llm.to(DEVICE)
model_llm.eval()

print(f'✅ {MODEL_NAME} loaded on {DEVICE}')
print(f'   Parameters : {sum(p.numel() for p in model_llm.parameters()) / 1e6:.0f}M')

Loading google/mt5-small...
This may take a few minutes on first run (downloading model weights).


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

✅ google/mt5-small loaded on cuda
   Parameters : 300M


In [16]:
import re

def build_prompt(question: str, language: str = None) -> str:
    if language:
        lang_name = subset_to_language_name(language)
        return f'Answer this health question in {lang_name}: {question}'
    return f'Answer this health question: {question}'


def generate_answers_batch(questions: list, languages: list = None,
                           batch_size: int = BATCH_SIZE_LLM) -> list:
    if languages is None:
        languages = [None] * len(questions)

    all_answers = []
    n_batches   = (len(questions) + batch_size - 1) // batch_size

    for batch_idx in range(n_batches):
        start = batch_idx * batch_size
        end   = min(start + batch_size, len(questions))

        batch_questions = questions[start:end]
        batch_languages = languages[start:end]

        prompts = [
            build_prompt(q, l)
            for q, l in zip(batch_questions, batch_languages)
        ]

        inputs = tokenizer(
            prompts,
            return_tensors = 'pt',
            padding        = True,
            truncation     = True,
            max_length     = MAX_INPUT_LENGTH,
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model_llm.generate(
                **inputs,
                max_new_tokens  = MAX_OUTPUT_LENGTH,
                num_beams       = NUM_BEAMS,
                early_stopping  = True,
                no_repeat_ngram_size = 3,
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        cleaned = [re.sub(r'<extra_id_\d+>', '', ans).strip() for ans in decoded]
        all_answers.extend(cleaned)

        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == n_batches:
            print(f'  Batch {batch_idx + 1}/{n_batches} — {end}/{len(questions)} questions processed')

    return all_answers

print('✅ LLM generation functions defined')

✅ LLM generation functions defined


In [17]:
# ── Quick sanity check on a few examples ──────────────────────────────────────
print('Running sanity check on 3 validation examples...')

sample     = val.head(3)
q_sample   = sample[QUESTION_COL].tolist()
l_sample   = sample[LANG_COL].tolist() if LANG_COL else None
ref_sample = sample[ANSWER_COL].tolist()

gen_sample = generate_answers_batch(q_sample, l_sample, batch_size=3)

for i, (q, ref, gen) in enumerate(zip(q_sample, ref_sample, gen_sample)):
    lang = l_sample[i] if l_sample else 'unknown'
    print(f'\n[{i+1}] Language : {lang}')
    print(f'    Question  : {q[:120]}')
    print(f'    Reference : {ref[:120]}')
    print(f'    Generated : {gen[:120]}')

Running sanity check on 3 validation examples...
  Batch 1/1 — 3/3 questions processed

[1] Language : Aka_Gha
    Question  : Sɛn na nwomasua ne adwuma nteteeɛ boa akuo a eye mmabun a wɔ hia neaɛma sokoronko ne ohaw ahorow, atubrafo, anaa wɔn a w
    Reference : Nhyehyɛeɛ aa ama ne mu so te sɛ senea aborɔfo ka no 'STEM' ne 'vocational training' se ɛbɛ adrɛse mmabun kuokuo ahohia s
    Generated : .

[2] Language : Aka_Gha
    Question  : Dɛn nti na ɛho hia sɛ mmabun te wɔn nna ne awo hokwan ahorow ase?
    Reference : Nna ne awo hokwan ahorow a wɔte ase no ma mmabun tumi: Si gyinae a ɛfata wɔ wɔn nipadua, nna, ne abusuabɔ ho. Kamfo wɔn 
    Generated : 

[3] Language : Aka_Gha
    Question  : Mɛyɛ dɛn atumi abɔ asisifo ho amanneɛ wɔ ɔkwan a etu mpɔn na ahobammɔ wom so, na anammɔn bɛn na metumi atu de ahwɛ ahu s
    Reference : Ayayade ho nsɛm a wɔbɛbɔ ho amanneɛ yiye na ahobammɔ wom no hwehwɛ sɛ wɔyɛ nneɛma a edidi so yi: Wɔkyerɛw ayayade no ho 
    Generated : .


In [18]:
# ── Validate LLM baseline on the local validation set ─────────────────────────
# Note: this cell can take several minutes depending on your hardware.
# Reduce VALIDATION_SAMPLE_SIZE to speed it up during experimentation.

VALIDATION_SAMPLE_SIZE = 200   # Set to None to evaluate on the full validation set

if VALIDATION_SAMPLE_SIZE:
    val_sample = val.sample(
        n            = min(VALIDATION_SAMPLE_SIZE, len(val)),
        random_state = SEED
    )
else:
    val_sample = val

print(f'Evaluating LLM baseline on {len(val_sample)} validation examples...')

val_questions = val_sample[QUESTION_COL].tolist()
val_languages = val_sample[LANG_COL].tolist() if LANG_COL else None
val_references= val_sample[ANSWER_COL].tolist()

val_predictions_llm = generate_answers_batch(val_questions, val_languages)

if compute_rouge:
    metrics_llm = compute_rouge(val_predictions_llm, val_references)
    print(f'\n📊 LLM Baseline — Validation ROUGE Scores ({MODEL_NAME})')
    print(f'   ROUGE-1 F1 : {metrics_llm["rouge1_f1"]:.4f}')
    print(f'   ROUGE-L F1 : {metrics_llm["rougeL_f1"]:.4f}')

    if LANG_COL and LANG_COL in val_sample.columns:
        print('\n📊 ROUGE scores by language (LLM baseline):')
        lang_metrics_llm = compute_rouge_by_language(
            val_predictions_llm,
            val_references,
            val_sample[LANG_COL].tolist()
        )
        display(lang_metrics_llm.round(4))

Evaluating LLM baseline on 200 validation examples...
  Batch 10/100 — 20/200 questions processed
  Batch 20/100 — 40/200 questions processed
  Batch 30/100 — 60/200 questions processed
  Batch 40/100 — 80/200 questions processed
  Batch 50/100 — 100/200 questions processed
  Batch 60/100 — 120/200 questions processed
  Batch 70/100 — 140/200 questions processed
  Batch 80/100 — 160/200 questions processed
  Batch 90/100 — 180/200 questions processed
  Batch 100/100 — 200/200 questions processed

📊 LLM Baseline — Validation ROUGE Scores (google/mt5-small)
   ROUGE-1 F1 : 0.0013
   ROUGE-L F1 : 0.0013

📊 ROUGE scores by language (LLM baseline):


,rouge1_f1,rougeL_f1
Aka_Gha,0.0011,0.0011
Amh_Eth,0.0111,0.0111
Eng_Eth,0.0000,0.0000
Eng_Gha,0.0000,0.0000
Eng_Ken,0.0000,0.0000
Eng_Uga,0.0000,0.0000
Lug_Uga,0.0000,0.0000
Swa_Ken,0.0000,0.0000


In [19]:
# ── Generate LLM predictions for the full test set ────────────────────────────
print(f'Generating LLM answers for {len(test)} test questions...')
print('This may take several minutes.')

test_questions_all = test[TEST_QUESTION_COL].tolist()
test_languages_all = test[TEST_LANG_COL].tolist() if TEST_LANG_COL else None

test_pred_llm = generate_answers_batch(test_questions_all, test_languages_all)

print(f'\n✅ Generated {len(test_pred_llm)} answers')

# Preview
preview_llm = test[[TEST_ID_COL, TEST_QUESTION_COL]].copy()
preview_llm['llm_answer'] = test_pred_llm
display(preview_llm.head(5))

Generating LLM answers for 2618 test questions...
This may take several minutes.
  Batch 10/1309 — 20/2618 questions processed
  Batch 20/1309 — 40/2618 questions processed
  Batch 30/1309 — 60/2618 questions processed
  Batch 40/1309 — 80/2618 questions processed
  Batch 50/1309 — 100/2618 questions processed
  Batch 60/1309 — 120/2618 questions processed
  Batch 70/1309 — 140/2618 questions processed
  Batch 80/1309 — 160/2618 questions processed
  Batch 90/1309 — 180/2618 questions processed
  Batch 100/1309 — 200/2618 questions processed
  Batch 110/1309 — 220/2618 questions processed
  Batch 120/1309 — 240/2618 questions processed
  Batch 130/1309 — 260/2618 questions processed
  Batch 140/1309 — 280/2618 questions processed
  Batch 150/1309 — 300/2618 questions processed
  Batch 160/1309 — 320/2618 questions processed
  Batch 170/1309 — 340/2618 questions processed
  Batch 180/1309 — 360/2618 questions processed
  Batch 190/1309 — 380/2618 questions processed
  Batch 200/1309 — 4

,ID,input,llm_answer
0,ID_TS_Aka_Gha_A3B1799D,"Fa nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow a wɔreyɛ adwuma de asiw GBV ano ma.",nneɛma
1,ID_TS_Aka_Gha_1C80317F,Dɛn ne nea ebetumi afi hokwan a mmabun wɔ sɛ wonya nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu a wobu...,?eловна
2,ID_TS_Aka_Gha_06671AD1,Akwan bɛn na mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi aba ɛfa nnipa a wɔbɛgyina ho kɛkɛ 'bystander' wɔ bere a as...,.
3,ID_TS_Aka_Gha_BDD640FB,"Sɛnea amammerɛ mu mmra, asetena mu suban, ne tumi mu nsakraeɛ a ɛwɔ nkuro ne atuhoamafoɔ mu ka mmaabun nkitahodi ne ...",.
4,ID_TS_Aka_Gha_46685257,Adɛn nti na mmara nsesaeɛ 'policy advocacy' ho hia ma mmabun nyin wɔ biribiara mu ?,


## 10 — Create Submission Files

Each submission must contain exactly four columns: `ID`, `TargetRLF1`, `TargetR1F1`, `TargetLLM`.
All three target columns should contain the same generated answer.

In [20]:
def make_submission(ids, predictions, output_path):
    """
    Build and save a valid Zindi submission file.

    Parameters
    ----------
    ids         : array-like of row IDs
    predictions : list[str] of generated answers
    output_path : str or Path
    """
    # Belt-and-suspenders: strip any residual sentinel tokens before saving.
    clean_preds = [re.sub(r'<extra_id_\d+>', '', str(p)).strip() for p in predictions]

    sub = pd.DataFrame()
    sub['ID']         = ids
    sub['TargetRLF1'] = clean_preds
    sub['TargetR1F1'] = clean_preds
    sub['TargetLLM']  = clean_preds

    sub = sub[['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']]

    # ── Submission checks ─────────────────────────────────────────────────
    required_cols = ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']
    assert list(sub.columns) == required_cols, \
        f'Expected columns {required_cols}, got {list(sub.columns)}'
    assert len(sub) == len(test), \
        f'Row count mismatch: {len(sub)} predictions vs {len(test)} test rows'
    assert sub[['TargetRLF1', 'TargetR1F1', 'TargetLLM']].notna().all().all(), \
        'Missing values found in submission'
    assert (sub['TargetRLF1'] == sub['TargetR1F1']).all(), \
        'TargetRLF1 and TargetR1F1 differ'
    assert (sub['TargetRLF1'] == sub['TargetLLM']).all(), \
        'TargetRLF1 and TargetLLM differ'

    sub.to_csv(output_path, index=False, encoding='utf-8')
    print(f'✅ Submission saved to: {output_path}')
    print(f'   Shape : {sub.shape}')
    display(sub.head(3))
    return sub


# ── Save TF-IDF submission ─────────────────────────────────────────────────────
print('Saving TF-IDF baseline submission...')
sub_tfidf = make_submission(test[TEST_ID_COL].values, test_pred_tfidf, OUTPUT_TFIDF)

print()

# ── Save LLM submission ────────────────────────────────────────────────────────
print('Saving LLM baseline submission...')
sub_llm = make_submission(test[TEST_ID_COL].values, test_pred_llm, OUTPUT_LLM)

Saving TF-IDF baseline submission...
✅ Submission saved to: /content/submission_tfidf_baseline.csv
   Shape : (2618, 4)


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa intanɛt so nneɛma a efi nnwumakuo ahorow a wogye wɔn di mu a ɛfa GBV n...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa intanɛt so nneɛma a efi nnwumakuo ahorow a wogye wɔn di mu a ɛfa GBV n...","Nhomasua Nneɛma: Hwehwɛ nhoma, nkratawa, anaa intanɛt so nneɛma a efi nnwumakuo ahorow a wogye wɔn di mu a ɛfa GBV n..."
1,ID_TS_Aka_Gha_1C80317F,"Yiw, mmabun betumi ahwehwɛ mmara kwan so mmoa sɛ wogye di sɛ wɔabu hokwan a wɔwɔ sɛ wodi wɔn ho so wɔ nipadua mu no ...","Yiw, mmabun betumi ahwehwɛ mmara kwan so mmoa sɛ wogye di sɛ wɔabu hokwan a wɔwɔ sɛ wodi wɔn ho so wɔ nipadua mu no ...","Yiw, mmabun betumi ahwehwɛ mmara kwan so mmoa sɛ wogye di sɛ wɔabu hokwan a wɔwɔ sɛ wodi wɔn ho so wɔ nipadua mu no ..."
2,ID_TS_Aka_Gha_06671AD1,Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basabasayɛ kwan denam mmoa a wɔde ma (bystanders) ma wohu/yɛ biribi no so. ...,Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basabasayɛ kwan denam mmoa a wɔde ma (bystanders) ma wohu/yɛ biribi no so. ...,Nnipa a wɔn ho nka ho tee yɛ biribi. Ɛsi basabasayɛ kwan denam mmoa a wɔde ma (bystanders) ma wohu/yɛ biribi no so. ...



Saving LLM baseline submission...
✅ Submission saved to: /content/submission_llm_baseline.csv
   Shape : (2618, 4)


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,nneɛma,nneɛma,nneɛma
1,ID_TS_Aka_Gha_1C80317F,?eловна,?eловна,?eловна
2,ID_TS_Aka_Gha_06671AD1,.,.,.


## 11 — Compare Baselines

In [21]:
if compute_rouge:
    # Recompute TF-IDF validation scores for comparison
    tfidf_preds_val, _, _ = answerer_valid.predict(
        val_sample, question_col=QUESTION_COL, group_col=GROUP_COL
    )
    metrics_tfidf = compute_rouge(tfidf_preds_val, val_references)

    comparison = pd.DataFrame({
        'Baseline'  : ['TF-IDF Retrieval', f'LLM ({MODEL_NAME})'],
        'ROUGE-1 F1': [metrics_tfidf['rouge1_f1'], metrics_llm['rouge1_f1']],
        'ROUGE-L F1': [metrics_tfidf['rougeL_f1'], metrics_llm['rougeL_f1']],
    })

    print('📊 Baseline Comparison (validation set):')
    display(comparison.round(4))
    print()
    print('Note: The LLM score shown here is for a zero-shot (untrained) model.')
    print('Fine-tuning on the training data will significantly improve this.')

📊 Baseline Comparison (validation set):


,Baseline,ROUGE-1 F1,ROUGE-L F1
0,TF-IDF Retrieval,0.4135,0.3581
1,LLM (google/mt5-small),0.0013,0.0013



Note: The LLM score shown here is for a zero-shot (untrained) model.
Fine-tuning on the training data will significantly improve this.


## 12 — Fine-tuning the LLM on Training Data

Fine-tuning adapts the pre-trained mT5 weights to the health QA task and all five languages. After training, answers are regenerated for the test set and a new submission file is saved.

**Key fixes vs. the original template:**
- Prompts are built with `build_prompt()` at training time — matching exactly what is used at inference
- Language names (resolved from `subset`) are included in training prompts
- Padding tokens in labels are masked to `-100` so the loss ignores them
- `DataCollatorForSeq2Seq` handles dynamic padding correctly
- A validation split monitors overfitting during training
- After `trainer.train()`, the test set is regenerated and a new submission is saved

In [22]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset

# ── Fine-tuning configuration ──────────────────────────────────────────────
FINETUNE_OUTPUT_DIR     = './mt5-small-exp6-langtag'
FINETUNE_EPOCHS         = 3
FINETUNE_BATCH_SIZE     = 2
FINETUNE_LEARNING_RATE  = 1e-4
FINETUNE_MAX_INPUT_LEN  = 128
FINETUNE_MAX_TARGET_LEN = 256
FINETUNE_VAL_SIZE       = 0.05
GRADIENT_ACCUM_STEPS    = 8

OUTPUT_FINETUNED = Path('/content/submission_exp6_langtag.csv')

print(f'Fine-tuning config:')
print(f'  Model            : {MODEL_NAME}')
print(f'  Epochs           : {FINETUNE_EPOCHS}')
print(f'  Batch size       : {FINETUNE_BATCH_SIZE}')
print(f'  Learning rate    : {FINETUNE_LEARNING_RATE}')
print(f'  Max input tokens : {FINETUNE_MAX_INPUT_LEN}')
print(f'  Max target tokens: {FINETUNE_MAX_TARGET_LEN}')
print(f'  Val split        : {FINETUNE_VAL_SIZE:.0%}')
print(f'  Output dir       : {FINETUNE_OUTPUT_DIR}')

Fine-tuning config:
  Model            : google/mt5-small
  Epochs           : 3
  Batch size       : 2
  Learning rate    : 0.0001
  Max input tokens : 128
  Max target tokens: 256
  Val split        : 5%
  Output dir       : ./mt5-small-exp6-langtag


In [23]:
# ── Build prompt-aware training dataset ───────────────────────────────────
# Critical: we use build_prompt() here so training inputs match inference
# inputs exactly. The language name (resolved from subset) is included so
# the model learns to condition its output on the target language.

def make_hf_dataset(df, question_col, answer_col, lang_col):
    """
    Convert a pandas DataFrame to a HuggingFace Dataset with prompt-formatted
    inputs and tokenised labels.

    Parameters
    ----------
    df           : pd.DataFrame
    question_col : str  — column containing the question text
    answer_col   : str  — column containing the reference answer
    lang_col     : str  — column containing the subset code (e.g. 'Amh_Eth')

    Returns
    -------
    datasets.Dataset with columns: input_ids, attention_mask, labels
    """
    records = []
    for _, row in df.iterrows():
        prompt = build_prompt(
            question = str(row[question_col]),
            language = str(row[lang_col]) if lang_col and lang_col in df.columns else None,
        )
        records.append({'prompt': prompt, 'answer': str(row[answer_col])})

    raw_ds = Dataset.from_list(records)

    def preprocess(examples):
        # Tokenise inputs (prompts)
        model_inputs = tokenizer(
            examples['prompt'],
            max_length  = FINETUNE_MAX_INPUT_LEN,
            truncation  = True,
            padding     = False,   # DataCollatorForSeq2Seq handles padding dynamically
        )
        # Tokenise targets (reference answers)
        # Use text_target= (the modern API). The older context manager
        # was removed in transformers >= 4.28 and must not be used.
        labels = tokenizer(
            text_target = examples['answer'],
            max_length  = FINETUNE_MAX_TARGET_LEN,
            truncation  = True,
            padding     = False,
        )
        # Mask padding tokens in labels so the loss ignores them.
        # Without this the model wastes capacity learning to predict pad tokens.
        label_ids = labels['input_ids']
        model_inputs['labels'] = [
            [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
            for seq in label_ids
        ]
        return model_inputs

    return raw_ds.map(preprocess, batched=True, remove_columns=['prompt', 'answer'])


# ── Split off a validation set from training data ──────────────────────────
from sklearn.model_selection import train_test_split

train_df, val_ft_df = train_test_split(
    train,
    test_size    = FINETUNE_VAL_SIZE,
    random_state = SEED,
    stratify     = train[LANG_COL] if LANG_COL in train.columns else None,
)

# Use 30% of training data for faster experimentation
train_df = train_df.sample(frac=0.3, random_state=42)
print(f'Using {len(train_df):,} training examples (30% subset)')

print(f'Fine-tuning split — train: {len(train_df):,}  val: {len(val_ft_df):,}')

hf_train_ds = make_hf_dataset(train_df,  QUESTION_COL, ANSWER_COL, LANG_COL)
hf_val_ds   = make_hf_dataset(val_ft_df, QUESTION_COL, ANSWER_COL, LANG_COL)

print(f'HF train dataset : {hf_train_ds}')
print(f'HF val dataset   : {hf_val_ds}')


Using 8,497 training examples (30% subset)
Fine-tuning split — train: 8,497  val: 1,491


Map:   0%|          | 0/8497 [00:00<?, ? examples/s]

Map:   0%|          | 0/1491 [00:00<?, ? examples/s]

HF train dataset : Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 8497
})
HF val dataset   : Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1491
})


In [24]:
# ── Data collator — handles dynamic padding and label masking ─────────────
data_collator = DataCollatorForSeq2Seq(
    tokenizer  = tokenizer,
    model      = model_llm,
    label_pad_token_id = -100,
    pad_to_multiple_of = 8,
)

# ── Training arguments ─────────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir                  = FINETUNE_OUTPUT_DIR,
    num_train_epochs            = FINETUNE_EPOCHS,
    per_device_train_batch_size = FINETUNE_BATCH_SIZE,
    per_device_eval_batch_size  = FINETUNE_BATCH_SIZE,
    gradient_accumulation_steps = 8,
    learning_rate               = FINETUNE_LEARNING_RATE,
    predict_with_generate       = True,
    bf16                        = (DEVICE == 'cuda' and torch.cuda.is_bf16_supported()),
    fp16                        = (DEVICE == 'cuda' and not torch.cuda.is_bf16_supported()),
    eval_strategy               = 'epoch',
    save_strategy               = 'no',
    load_best_model_at_end      = False,
    logging_steps               = 100,
    generation_max_length       = FINETUNE_MAX_TARGET_LEN,
    report_to                   = 'none',
)

# ── Trainer ────────────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model           = model_llm,
    args            = training_args,
    train_dataset   = hf_train_ds,
    eval_dataset    = hf_val_ds,
    processing_class = tokenizer,
    data_collator   = data_collator,
)

print('Starting fine-tuning...')
print(f'  Training on {len(hf_train_ds):,} examples for {FINETUNE_EPOCHS} epoch(s)')
print(f'  Validating on {len(hf_val_ds):,} examples after each epoch')

trainer.train()

print('\n✅ Fine-tuning complete')
print(f'   Best checkpoint saved to: {FINETUNE_OUTPUT_DIR}')

Starting fine-tuning...
  Training on 8,497 examples for 3 epoch(s)
  Validating on 1,491 examples after each epoch


Epoch,Training Loss,Validation Loss
1,32.150398,3.106998
2,29.487734,2.928356
3,28.616821,2.879843



✅ Fine-tuning complete
   Best checkpoint saved to: ./mt5-small-exp6-langtag


In [25]:
# ── Regenerate test predictions with the fine-tuned model ─────────────────
# model_llm now holds the best fine-tuned checkpoint (load_best_model_at_end=True).
# We reuse generate_answers_batch() directly — it already uses model_llm
# and applies the same build_prompt() + sentinel-token cleanup.

print(f'Generating fine-tuned answers for {len(test):,} test questions...')
model_llm.eval()

test_questions_ft = test[TEST_QUESTION_COL].tolist()
test_languages_ft = test[TEST_LANG_COL].tolist() if TEST_LANG_COL else None

test_pred_finetuned = generate_answers_batch(test_questions_ft, test_languages_ft)

print(f'\n✅ Generated {len(test_pred_finetuned):,} answers')

# Preview
preview_ft = test[[TEST_ID_COL, TEST_QUESTION_COL]].copy()
preview_ft['finetuned_answer'] = test_pred_finetuned
display(preview_ft.head(5))

# ── Validate on val set before saving ─────────────────────────────────────
if compute_rouge:
    val_q_ft  = val[QUESTION_COL].tolist()
    val_l_ft  = val[LANG_COL].tolist() if LANG_COL else None
    val_ref_ft = val[ANSWER_COL].tolist()

    val_pred_ft = generate_answers_batch(val_q_ft, val_l_ft)
    metrics_ft  = compute_rouge(val_pred_ft, val_ref_ft)

    print(f'\n📊 Fine-tuned LLM — Validation ROUGE Scores')
    print(f'   ROUGE-1 F1 : {metrics_ft["rouge1_f1"]:.4f}')
    print(f'   ROUGE-L F1 : {metrics_ft["rougeL_f1"]:.4f}')

    if LANG_COL and LANG_COL in val.columns:
        print('\n📊 ROUGE scores by language (fine-tuned model):')
        lang_metrics_ft = compute_rouge_by_language(
            val_pred_ft, val_ref_ft, val[LANG_COL].tolist()
        )
        display(lang_metrics_ft.round(4))

# ── Save fine-tuned submission ─────────────────────────────────────────────
print('\nSaving fine-tuned submission...')
sub_finetuned = make_submission(
    ids         = test[TEST_ID_COL].values,
    predictions = test_pred_finetuned,
    output_path = OUTPUT_FINETUNED,
)


Generating fine-tuned answers for 2,618 test questions...
  Batch 10/1309 — 20/2618 questions processed
  Batch 20/1309 — 40/2618 questions processed
  Batch 30/1309 — 60/2618 questions processed
  Batch 40/1309 — 80/2618 questions processed
  Batch 50/1309 — 100/2618 questions processed
  Batch 60/1309 — 120/2618 questions processed
  Batch 70/1309 — 140/2618 questions processed
  Batch 80/1309 — 160/2618 questions processed
  Batch 90/1309 — 180/2618 questions processed
  Batch 100/1309 — 200/2618 questions processed
  Batch 110/1309 — 220/2618 questions processed
  Batch 120/1309 — 240/2618 questions processed
  Batch 130/1309 — 260/2618 questions processed
  Batch 140/1309 — 280/2618 questions processed
  Batch 150/1309 — 300/2618 questions processed
  Batch 160/1309 — 320/2618 questions processed
  Batch 170/1309 — 340/2618 questions processed
  Batch 180/1309 — 360/2618 questions processed
  Batch 190/1309 — 380/2618 questions processed
  Batch 200/1309 — 400/2618 questions proce

,ID,input,finetuned_answer
0,ID_TS_Aka_Gha_A3B1799D,"Fa nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumayɛbea ahorow, ne akuo ahorow a wɔreyɛ adwuma de asiw GBV ano ma.","This is a nneɛma a wɔde bɛyɛ adwumayɛbea ahorow, ne akuo ahorow a wode nkyerɛkyerɛ nnaɛma."
1,ID_TS_Aka_Gha_1C80317F,Dɛn ne nea ebetumi afi hokwan a mmabun wɔ sɛ wonya nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu a wobu...,"This is a question about, Nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu."
2,ID_TS_Aka_Gha_06671AD1,Akwan bɛn na mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi aba ɛfa nnipa a wɔbɛgyina ho kɛkɛ 'bystander' wɔ bere a as...,"Yes, mmabun bɛtumi afa so ehunu nsusuanso a ɛfa nnipa a wɔbɛgyina ho kɛkɛ 'bystander' ne nneɛma bi mu ayɛ den."
3,ID_TS_Aka_Gha_BDD640FB,"Sɛnea amammerɛ mu mmra, asetena mu suban, ne tumi mu nsakraeɛ a ɛwɔ nkuro ne atuhoamafoɔ mu ka mmaabun nkitahodi ne ...","Ammerɛ mu mmra, asetena mu suban, ne tumi mu nsakraeɛ a ɛwɔ nkitahodi ne akannifo nnwuma wɔ ASRH."
4,ID_TS_Aka_Gha_46685257,Adɛn nti na mmara nsesaeɛ 'policy advocacy' ho hia ma mmabun nyin wɔ biribiara mu ?,"Yes, nsesaeɛ 'policy advocacy' ho hia ma mmabun nyin wɔ biribiara mu."


  Batch 10/3343 — 20/6686 questions processed
  Batch 20/3343 — 40/6686 questions processed
  Batch 30/3343 — 60/6686 questions processed
  Batch 40/3343 — 80/6686 questions processed
  Batch 50/3343 — 100/6686 questions processed
  Batch 60/3343 — 120/6686 questions processed
  Batch 70/3343 — 140/6686 questions processed
  Batch 80/3343 — 160/6686 questions processed
  Batch 90/3343 — 180/6686 questions processed
  Batch 100/3343 — 200/6686 questions processed
  Batch 110/3343 — 220/6686 questions processed
  Batch 120/3343 — 240/6686 questions processed
  Batch 130/3343 — 260/6686 questions processed
  Batch 140/3343 — 280/6686 questions processed
  Batch 150/3343 — 300/6686 questions processed
  Batch 160/3343 — 320/6686 questions processed
  Batch 170/3343 — 340/6686 questions processed
  Batch 180/3343 — 360/6686 questions processed
  Batch 190/3343 — 380/6686 questions processed
  Batch 200/3343 — 400/6686 questions processed
  Batch 210/3343 — 420/6686 questions processed
  Bat

,rouge1_f1,rougeL_f1
Aka_Gha,0.1745,0.1411
Amh_Eth,0.0643,0.0614
Eng_Eth,0.2533,0.2495
Eng_Gha,0.1257,0.1107
Eng_Ken,0.0534,0.0471
Eng_Uga,0.0478,0.0426
Lug_Uga,0.0532,0.0494
Swa_Ken,0.0698,0.0601



Saving fine-tuned submission...
✅ Submission saved to: /content/submission_exp6_langtag.csv
   Shape : (2618, 4)


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"This is a nneɛma a wɔde bɛyɛ adwumayɛbea ahorow, ne akuo ahorow a wode nkyerɛkyerɛ nnaɛma.","This is a nneɛma a wɔde bɛyɛ adwumayɛbea ahorow, ne akuo ahorow a wode nkyerɛkyerɛ nnaɛma.","This is a nneɛma a wɔde bɛyɛ adwumayɛbea ahorow, ne akuo ahorow a wode nkyerɛkyerɛ nnaɛma."
1,ID_TS_Aka_Gha_1C80317F,"This is a question about, Nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu.","This is a question about, Nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu.","This is a question about, Nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu."
2,ID_TS_Aka_Gha_06671AD1,"Yes, mmabun bɛtumi afa so ehunu nsusuanso a ɛfa nnipa a wɔbɛgyina ho kɛkɛ 'bystander' ne nneɛma bi mu ayɛ den.","Yes, mmabun bɛtumi afa so ehunu nsusuanso a ɛfa nnipa a wɔbɛgyina ho kɛkɛ 'bystander' ne nneɛma bi mu ayɛ den.","Yes, mmabun bɛtumi afa so ehunu nsusuanso a ɛfa nnipa a wɔbɛgyina ho kɛkɛ 'bystander' ne nneɛma bi mu ayɛ den."
